In [1]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### Load Dataset

In [11]:
import random
from torch.utils.data import Subset

data_dir = "./imagenet_validation"      # Download from https://www.kaggle.com/datasets/tusonggao/imagenet-validation-dataset
synset2label = {}
with open(data_dir + "/labels.txt", "r") as f:
    for line in f:
        synset, label = line.strip().split(" ", 1)
        synset2label[synset] = label

transform = transforms.Compose([
    transforms.Resize(256),
    # transforms.CenterCrop(224),
    transforms.CenterCrop(256),
    transforms.ToTensor(),  # just brings it to [0, 1] float tensor
])
dataset = datasets.ImageFolder(root=data_dir, transform=transform)

clip_labels = []
for synset, idx in sorted(dataset.class_to_idx.items(), key=lambda x: x[1]):
    label = synset2label[synset]
    clip_labels.append(f"a photo of a {label}")

siglip_labels = []
for synset, idx in sorted(dataset.class_to_idx.items(), key=lambda x: x[1]):
    label = synset2label[synset]
    siglip_labels.append(f"This is a photo of a {label}")

n_samples = 5000
indices = random.sample(range(len(dataset)), n_samples)
subset = Subset(dataset, indices)

loader = DataLoader(
    subset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
)

num_classes = len(dataset.classes)
print("Dataset size:", len(loader.dataset))
print("Classes:", num_classes)

Dataset size: 5000
Classes: 1000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


# Evaluate Models

In [12]:
from transformers import CLIPModel, CLIPProcessor, AutoModel, AutoProcessor


class ModelWrapper(nn.Module):
    def __init__(self, model_name, num_classes, device):
        super().__init__()
        self.name = model_name
        self.device = device

        # ===== CLIP =====
        if model_name == "clip_base":
            self.model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16")
            self.processor = CLIPProcessor.from_pretrained(
                "openai/clip-vit-base-patch16"
            )
            self.labels = clip_labels
        elif model_name == "clip_large":
            self.model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
            self.processor = CLIPProcessor.from_pretrained(
                "openai/clip-vit-large-patch14"
            )
            self.labels = clip_labels

        # ===== SigLIP =====
        elif model_name == "siglip_base":
            self.model = AutoModel.from_pretrained("google/siglip-base-patch16-224")
            self.processor = AutoProcessor.from_pretrained(
                "google/siglip-base-patch16-224"
            )
            self.labels = siglip_labels
        elif model_name == "siglip_large":
            self.model = AutoModel.from_pretrained("google/siglip-large-patch16-256")
            self.processor = AutoProcessor.from_pretrained(
                "google/siglip-large-patch16-256"
            )
            self.labels = siglip_labels

        else:
            raise ValueError(f"Unknown model: {model_name}")

        self.model = self.model.to(device)
        self.eval()

        with torch.no_grad():
            text_inputs = self.processor(
                text=self.labels, return_tensors="pt", padding=True
            ).to(device)

            self.text_features = self.model.get_text_features(
                **text_inputs
            ).pooler_output
            self.text_features = self.text_features / self.text_features.norm(
                dim=-1, keepdim=True
            )

    def forward(self, images):
        with torch.no_grad():
            image_inputs = self.processor(
                images=images, return_tensors="pt", do_rescale=False
            ).to(self.device)
            image_features = self.model.get_image_features(**image_inputs).pooler_output
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

            logits = image_features @ self.text_features.T
            return logits

In [13]:
from pathlib import Path
import torchvision.transforms.functional as TF


def apply_affine_batch(images, angle, translate, scale, shear):
    return torch.stack(
        [
            TF.affine(img, angle=angle, translate=translate, scale=scale, shear=shear)
            for img in images
        ]
    )


def evaluate(
    model,
    loader,
    model_name: str = "Model",
    translation: tuple[float, float] = [0, 0],
    rotation: float = 0.0,
    scale: float = 1.0,
    shear: tuple[float, float] = [0, 0],
    save_results_path: str = "results.csv",
):
    correct_top1 = 0
    correct_top5 = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(loader):
            images = apply_affine_batch(
                images, angle=rotation, translate=translation, scale=scale, shear=shear
            )

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            preds_top1 = outputs.argmax(dim=1)
            correct_top1 += (preds_top1 == labels).sum().item()

            _, preds_top5 = outputs.topk(5, dim=1)
            correct_top5 += (preds_top5 == labels.unsqueeze(1)).any(dim=1).sum().item()

            total += labels.size(0)

    acc_top1 = correct_top1 / total
    acc_top5 = correct_top5 / total

    results_path = Path(save_results_path)
    header = (
        "Model Name,Dataset Size,Translation X, Translation Y,"
        "Rotation,Scale,Shear X, Shear Y,Top-1 Accuracy,Top-5 Accuracy"
    )

    if not results_path.exists() or results_path.stat().st_size == 0:
        # file doesn't exist or is empty -> create with header
        with results_path.open("w", encoding="utf-8") as f:
            f.write(header + "\n")
    else:
        # file exists, check header consistency
        with results_path.open("r", encoding="utf-8") as f:
            first_line = f.readline().strip()
        if first_line != header:
            with results_path.open("w", encoding="utf-8") as f:
                f.write(header + "\n")

    # append result row
    with results_path.open("a", encoding="utf-8") as f:
        f.write(
            f"{model_name},{len(loader.dataset)},"
            f"{translation[0]},{translation[1]},{rotation},{scale},"
            f"{shear[0]},{shear[1]},{acc_top1:.4f},{acc_top5:.4f}\n"
        )

    return acc_top1, acc_top5

In [14]:
def evaluate_architecture(model_name, loader, geometric_transforms):
    model = ModelWrapper(model_name, num_classes, device)

    # Baseline evaluation without transformations
    if geometric_transforms["baseline"]:
        acc_top1, acc_top5 = evaluate(model, loader, model_name)
        print(f"Baseline (No Transformations):")
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

    # Translations
    for translation in geometric_transforms["translations"]:
        acc_top1, acc_top5 = evaluate(model, loader, model_name, translation=translation)
        print(f"Translation {translation}:")
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

    # Rotations
    for rotation in geometric_transforms["rotations"]:
        acc_top1, acc_top5 = evaluate(model, loader, model_name, rotation=rotation)
        print(f"Rotation {rotation}:")
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

    # Scales
    for scale in geometric_transforms["scales"]:
        acc_top1, acc_top5 = evaluate(model, loader, model_name, scale=scale)
        print(f"Scale {scale}:")
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

    # Shears
    for shear in geometric_transforms["shears"]:
        acc_top1, acc_top5 = evaluate(model, loader, model_name, shear=shear)
        print(f"Shear {shear}:")
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

### CLIP ViT Base

In [7]:
geometric_transforms = {
    "baseline": True,
    "translations": [
        (25, 0),  # Translate 25 pixels along X-axis
        (0, 25),  # Translate 25 pixels along Y-axis
        (-25, 0),  # Translate -25 pixels along X-axis
        (0, -25),  # Translate -25 pixels along Y-axis
        (50, 0),  # Translate 50 pixels along X-axis
        (0, 50),  # Translate 50 pixels along Y-axis
        (-50, 0),  # Translate -50 pixels along X-axis
        (0, -50),  # Translate -50 pixels along Y-axis
    ],
    "rotations": [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330],
    "scales": [0.5, 1.5, 2.0],
    "shears": [
        (25, 0),  # Shear 25 degrees along X-axis
        (0, 25),  # Shear 25 degrees along Y-axis
        (-25, 0),  # Shear -25 degrees along X-axis
        (0, -25),  # Shear -25 degrees along Y-axis
        (50, 0),  # Shear 50 degrees along X-axis
        (0, 50),  # Shear 50 degrees along Y-axis
        (-50, 0),  # Shear -50 degrees along X-axis
        (0, -50),  # Shear -50 degrees along Y-axis
    ],
}

evaluate_architecture("clip_base", loader, geometric_transforms)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

...

Shear (0, 50):
Top-1 Accuracy: 31.32%
Top-5 Accuracy: 57.76%


100%|██████████| 79/79 [01:24<00:00,  1.07s/it]


Shear (-50, 0):
Top-1 Accuracy: 31.50%
Top-5 Accuracy: 57.30%


100%|██████████| 79/79 [01:24<00:00,  1.07s/it]

Shear (0, -50):
Top-1 Accuracy: 31.72%
Top-5 Accuracy: 57.38%


### CLIP ViT Large

In [9]:
geometric_transforms = {
    "baseline": True,
    "translations": [
        (25, 0),  # Translate 25 pixels along X-axis
        (0, 25),  # Translate 25 pixels along Y-axis
        (-25, 0),  # Translate -25 pixels along X-axis
        (0, -25),  # Translate -25 pixels along Y-axis
        (50, 0),  # Translate 50 pixels along X-axis
        (0, 50),  # Translate 50 pixels along Y-axis
        (-50, 0),  # Translate -50 pixels along X-axis
        (0, -50),  # Translate -50 pixels along Y-axis
    ],
    "rotations": [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330],
    "scales": [0.5, 1.5, 2.0],
    "shears": [
        (25, 0),  # Shear 25 degrees along X-axis
        (0, 25),  # Shear 25 degrees along Y-axis
        (-25, 0),  # Shear -25 degrees along X-axis
        (0, -25),  # Shear -25 degrees along Y-axis
        (50, 0),  # Shear 50 degrees along X-axis
        (0, 50),  # Shear 50 degrees along Y-axis
        (-50, 0),  # Shear -50 degrees along X-axis
        (0, -50),  # Shear -50 degrees along Y-axis
    ],
}

evaluate_architecture("clip_large", loader, geometric_transforms)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

Shear (0, 50):
Top-1 Accuracy: 46.40%
Top-5 Accuracy: 71.10%


100%|██████████| 79/79 [04:41<00:00,  3.57s/it]


Shear (-50, 0):
Top-1 Accuracy: 45.50%
Top-5 Accuracy: 71.00%


100%|██████████| 79/79 [04:46<00:00,  3.62s/it]

Shear (0, -50):
Top-1 Accuracy: 46.28%
Top-5 Accuracy: 71.56%


### SigLIP Base

In [8]:
geometric_transforms = {
    "baseline": True,
    "translations": [
        (25, 0),  # Translate 25 pixels along X-axis
        (0, 25),  # Translate 25 pixels along Y-axis
        (-25, 0),  # Translate -25 pixels along X-axis
        (0, -25),  # Translate -25 pixels along Y-axis
        (50, 0),  # Translate 50 pixels along X-axis
        (0, 50),  # Translate 50 pixels along Y-axis
        (-50, 0),  # Translate -50 pixels along X-axis
        (0, -50),  # Translate -50 pixels along Y-axis
    ],
    "rotations": [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330],
    "scales": [0.5, 1.5, 2.0],
    "shears": [
        (25, 0),  # Shear 25 degrees along X-axis
        (0, 25),  # Shear 25 degrees along Y-axis
        (-25, 0),  # Shear -25 degrees along X-axis
        (0, -25),  # Shear -25 degrees along Y-axis
        (50, 0),  # Shear 50 degrees along X-axis
        (0, 50),  # Shear 50 degrees along Y-axis
        (-50, 0),  # Shear -50 degrees along X-axis
        (0, -50),  # Shear -50 degrees along Y-axis
    ],
}

evaluate_architecture("siglip_base", loader, geometric_transforms)

config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/813M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

Shear (0, 50):
Top-1 Accuracy: 23.98%
Top-5 Accuracy: 43.74%


100%|██████████| 79/79 [01:26<00:00,  1.09s/it]


Shear (-50, 0):
Top-1 Accuracy: 22.38%
Top-5 Accuracy: 41.06%


100%|██████████| 79/79 [01:27<00:00,  1.11s/it]

Shear (0, -50):
Top-1 Accuracy: 23.34%
Top-5 Accuracy: 42.60%


### SigLIP Large

In [15]:
geometric_transforms = {
    "baseline": True,
    "translations": [
        (25, 0),  # Translate 25 pixels along X-axis
        (0, 25),  # Translate 25 pixels along Y-axis
        (-25, 0),  # Translate -25 pixels along X-axis
        (0, -25),  # Translate -25 pixels along Y-axis
        (50, 0),  # Translate 50 pixels along X-axis
        (0, 50),  # Translate 50 pixels along Y-axis
        (-50, 0),  # Translate -50 pixels along X-axis
        (0, -50),  # Translate -50 pixels along Y-axis
    ],
    "rotations": [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330],
    "scales": [0.5, 1.5, 2.0],
    "shears": [
        (25, 0),  # Shear 25 degrees along X-axis
        (0, 25),  # Shear 25 degrees along Y-axis
        (-25, 0),  # Shear -25 degrees along X-axis
        (0, -25),  # Shear -25 degrees along Y-axis
        (50, 0),  # Shear 50 degrees along X-axis
        (0, 50),  # Shear 50 degrees along Y-axis
        (-50, 0),  # Shear -50 degrees along X-axis
        (0, -50),  # Shear -50 degrees along Y-axis
    ],
}

evaluate_architecture("siglip_large", loader, geometric_transforms)

config.json:   0%|          | 0.00/554 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/792 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/711 [00:00<?, ?B/s]

...

Translation (25, 0):
Top-1 Accuracy: 23.76%
Top-5 Accuracy: 40.72%


100%|██████████| 79/79 [04:52<00:00,  3.70s/it]


Translation (0, 25):
Top-1 Accuracy: 24.12%
Top-5 Accuracy: 41.16%


 28%|██▊       | 22/79 [01:27<03:46,  3.97s/it]


KeyboardInterrupt: 